# RAG 2
Multiple data source (PDF: Apple_form_10K.pdf, Excel: Sample_Financial_Data.xlsx)

In [1]:
from pathlib import Path
import random
from pydantic import BaseModel
import json

import pymupdf4llm
import pandas as pd 
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import openai
from rag_eval import RAGEvaluator

In [2]:
apple_doc_path = list(Path.cwd().rglob("Apple_form_10k.pdf"))
financial_data_path = list(Path.cwd().rglob("Sample_Financial_Data.xlsx"))

### Parsing and Chunking of PDF

> **NOTE:** As we want to add metadata to our chunks, so will be wrapping our chunks LangChain's **Document** class and add metadata like the file name and file type.

In [3]:
md_text = pymupdf4llm.to_markdown(str(apple_doc_path[0]))

In [4]:
# Chunking
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
pdf_chunks = text_splitter.split_text(md_text)

In [5]:
pdf_chunks[0]

Document(metadata={'Header 1': '**UNITED STATES SECURITIES AND EXCHANGE COMMISSION**'}, page_content='**Washington, D.C. 20549**')

In [6]:
# Since the splitter alreadt wraps the chunks in Document class, we can add more metadata to each chunk

pdf_chunk_ids = [f"pdf_chunk_{i}" for i in range(len(pdf_chunks))]

for index, chunks in enumerate(pdf_chunks):
	chunks.metadata["source"] = "apple_10k.md"
	chunks.metadata["doc_type"] = "PDF-Markdown"
	chunks.metadata["chunk_id"] = pdf_chunk_ids[index]

In [7]:
pdf_chunks[100].metadata

{'Header 1': 'For the fiscal year ended September 27, 2025',
 'Header 2': '_Income Taxes_',
 'source': 'apple_10k.md',
 'doc_type': 'PDF-Markdown',
 'chunk_id': 'pdf_chunk_100'}

### Parsing and Chunking of Excel 

In [8]:
financial_data = pd.read_excel(str(financial_data_path[0]))

In [9]:
financial_data.head()

,Segment,Country,Product,Discount Band,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit,Date,Month Number,Month Name,Year
0,Government,Canada,Carretera,NaN,1618.5,3,20,32370.0,0.0,32370.0,16185.0,16185.0,2014-01-01,1,January,2014
1,Government,Germany,Carretera,NaN,1321.0,3,20,26420.0,0.0,26420.0,13210.0,13210.0,2014-01-01,1,January,2014
2,Midmarket,France,Carretera,NaN,2178.0,3,15,32670.0,0.0,32670.0,21780.0,10890.0,2014-06-01,6,June,2014
3,Midmarket,Germany,Carretera,NaN,888.0,3,15,13320.0,0.0,13320.0,8880.0,4440.0,2014-06-01,6,June,2014
4,Midmarket,Mexico,Carretera,NaN,2470.0,3,15,37050.0,0.0,37050.0,24700.0,12350.0,2014-06-01,6,June,2014


In [10]:
# Row-as-a-string method
excel_chunks = []

excel_chunk_ids = [f"excel_chunk_{i}" for i in range(len(financial_data))]

for index, row in financial_data.iterrows():
	row_text = (
			f"In {row['Month Name']} {row['Year']}, the {row['Segment']} segment in {row['Country']} "
			f"purchased {row['Units Sold']:,} units of the product '{row['Product']}'. "
			f"The manufacturing price was ${row['Manufacturing Price']} and the sale price was ${row['Sale Price']}. "
			f"This transaction generated ${row['Gross Sales']:,} in gross sales with a discount band of ${row['Discount Band']} "
			f"amounting to ${row['Discounts']:,} in total discounts. Net sales were ${row[' Sales']}. "
			f"The Cost of Goods Sold (COGS) was ${row['COGS']:,}, resulting in a net profit of ${row['Profit']:,}."
		)
	
	doc = Document(
			page_content=row_text,
			metadata={
					"source": "financial_data.xlsx",
					"row_index": index,
					"doc_type": "Excel",
					"country": row['Country'],
					"product": row['Product'],
					"chunk_id": excel_chunk_ids[index]
			}
		)

	excel_chunks.append(doc)

In [11]:
excel_chunks[45].metadata

{'source': 'financial_data.xlsx',
 'row_index': 45,
 'doc_type': 'Excel',
 'country': 'France',
 'product': 'Amarilla',
 'chunk_id': 'excel_chunk_45'}

### Embedding in V DB

In [12]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [13]:
pdf_chunk_ids = [f"pdf_chunk_{i}" for i in range(len(pdf_chunks))]
excel_chunk_ids = [f"excel_chunk_{i}" for i in range(len(excel_chunks))]

In [14]:
db_name = Path.cwd() / "vdb" 

if db_name.exists():
    Chroma(persist_directory=str(db_name), embedding_function=embeddings).delete_collection()

vector_store = Chroma(
    collection_name="embeddings",
    embedding_function=embeddings,
    persist_directory=str(db_name)
)

vector_store.add_documents(documents=pdf_chunks, ids=pdf_chunk_ids)
vector_store.add_documents(documents=excel_chunks, ids=excel_chunk_ids)

print(f"Vectorstore created with {vector_store._collection.count()} documents")

Vectorstore created with 917 documents


### Retrieval

In [15]:
def context_building(context: str) -> str:
	extended_prompt = f"""You are an smart AI assistant helps in reading form 10k and financial data. You will answer the question in explainatory way.
	If relevant, use the given context to answer any question.
	If you don't know the answer, say so.
	Context:
	{context}"""
	return extended_prompt

In [16]:
def rag_pipeline(query: str) -> str:
	retriever = vector_store.similarity_search(query=query, k=3)
	context = ['\n\n' + i.page_content for i in retriever]
	extended_prompt = context_building(context)
	result = openai.chat.completions.create(
		model="gpt-4o-mini",
		messages=[{'role': 'system', 'content': extended_prompt}, {'role': 'user', 'content': query}]
	)
	result = result.choices[0].message.content
	data = {
		"question": query,
		"generated_answer": result,
		"contexts": [i.page_content for i in retriever],
		"chunk_used": [i.id for i in retriever]
	}
	return data
	

In [17]:
rag_pipeline("What were the total government sales in Canada for January?")

{'question': 'What were the total government sales in Canada for January?',
 'generated_answer': "In January 2014, the Government segment in Canada made a purchase of 1,618.5 units of the product 'Carretera'. This transaction generated gross sales of $32,370.0 with net sales also amounting to $32,370.0 since there were no total discounts applied. Therefore, the total government sales in Canada for January 2014 were $32,370.0.",
 'contexts': ["In January 2014, the Government segment in Canada purchased 1,618.5 units of the product 'Carretera'. The manufacturing price was $3 and the sale price was $20. This transaction generated $32,370.0 in gross sales with a discount band of $nan amounting to $0.0 in total discounts. Net sales were $32370.0. The Cost of Goods Sold (COGS) was $16,185.0, resulting in a net profit of $16,185.0.",
  "In September 2014, the Government segment in Canada purchased 707.0 units of the product 'Amarilla'. The manufacturing price was $260 and the sale price was $

## Evals

### Creating golden dataset 
Creating a synthetic golden dataset with multiple chunks  

In [18]:
class CoreFormat(BaseModel):
	question: str
	answer: str

In [19]:
gd_prompt = """
You are a QA engineer building a complex, multi-chunk evaluation dataset for a RAG system.
You will be provided with a list of text chunks retrieved from different sections or files.

Your task:
1. Review all the provided context chunks.
2. Synthesize the information to create ONE high-quality question that requires connecting the facts across at least 2 or 3 of these chunks to answer completely. 
3. Provide the comprehensive ground truth answer.

Output your response strictly as a JSON object with keys: "question" and "ground_truth_answer". No markdown wrappers.
"""

In [20]:
gd_path = str(Path.cwd() / "eval" / "golden_dataset.jsonl")

def golden_dataset_generator(n: int) -> list:
	for i in range(n):
		# random_chunks = random.choices(pdf_chunks + excel_chunks, k=3)   # For mixed chunks (pdf + excel)
		random_chunks = random.choices(pdf_chunks, k=3)   # For pdf chunks only
		
		serialized_context = ""
		for idx, chunk in enumerate(random_chunks):
			serialized_context += f"--- Chunk {idx+1} [Source: {chunk.metadata.get('source')}] ---\n"
			serialized_context += f"{chunk.page_content}\n\n"

		result = openai.chat.completions.parse(
			model="gpt-4o-mini",
			messages=[
				{'role': 'system', 'content': gd_prompt}, 
				{'role': 'user', 'content': f"Here are the retrieved chunks: {serialized_context}"}
				],
				response_format=CoreFormat
		)

		qa = result.choices[0].message.parsed	

		gd_row = {
			"question": qa.question,
			"expected_answer": qa.answer,
			"chunk_ids": [i.metadata['chunk_id'] for i in random_chunks],
		}

		with open(gd_path, "a", encoding="utf-8") as f:
			f.write(json.dumps(gd_row, ensure_ascii=False) + "\n")	

	return gd_row

In [21]:
a = golden_dataset_generator(20)

### Evaluation of retrieval

In [22]:
retrieval_test_results = []

with open(gd_path, 'r', encoding='utf-8') as f:
	for line in f:
		row = json.loads(line)
		question = row['question']

		retrieved = vector_store.similarity_search(question, k=3)

		result = {
			'question': question,
			'relevent_chunks': row['chunk_ids'],
			'retrieved_chunks': [i.id for i in retrieved]
			}

		retrieval_test_results.append(result)

		retrieval_test_results_path = Path.cwd() / 'eval' / 'retrieval_test_results.jsonl'
		
		with open(str(retrieval_test_results_path), 'a', encoding='utf-8') as f:
			f.write(json.dumps(result, ensure_ascii=False) + "\n")

In [24]:
retrieval_test_results[10]

{'question': "What was Apple's total provision for income taxes in 2025, and how does it compare to the computed expected tax for that year based on the statutory federal income tax rate?",
 'relevent_chunks': ['pdf_chunk_138', 'pdf_chunk_26', 'pdf_chunk_40'],
 'retrieved_chunks': ['pdf_chunk_138', 'pdf_chunk_90', 'pdf_chunk_53']}

In [25]:
report = RAGEvaluator.from_dict_list(retrieval_test_results, k=3)
RAGEvaluator(k=3).report(report)


  RAG RETRIEVAL EVALUATION REPORT  (k=3)
  Metric                    Score
  ------------------------------
  Hit Rate                 0.8500
  MRR                      0.7833
  Precision@k              0.3667
  Recall@k                 0.3667
  NDCG@k                   0.4464
  Queries evaluated : 20
  Queries with issues: 20

  FAILED / UNDERPERFORMING QUERIES

  Query ID : How do cybersecurity incidents and supply chain vulnerabilities impact the financial condition and stock price of the Company, as indicated in multiple sections of the retrieved chunks?
  Scores   → Hit:1.00 MRR:1.00 P:0.67 R:0.67 NDCG:0.77
  ⚠  INCOMPLETE RECALL (0.67) — Missed 1 relevant chunk(s): {'pdf_chunk_196'}

  Query ID : What factors contributed to Apple's lower effective tax rate for 2025 compared to the statutory federal income tax rate and the prior year’s effective tax rate?
  Scores   → Hit:1.00 MRR:1.00 P:0.33 R:0.33 NDCG:0.47
  ⚠  INCOMPLETE RECALL (0.33) — Missed 2 relevant chunk(s): {'pdf_chunk